# Pruebas estadísticas pareadas (Wilcoxon) de recuperación y generación

Recalcula, a partir de los archivos de evaluación ya generados, todas las pruebas de Wilcoxon que se citan en el documento del trabajo de grado, para que cada cifra tenga un archivo que la respalde.

| Bloque | Qué compara | Dónde se usa en el documento | Datos de entrada |
|---|---|---|---|
| 1. Recuperación | Configuraciones de retrieve, sobre el promedio (precisión + recall) / 2 por pregunta, n = 296 | Decisión de recuperación (Tabla 12) e hipótesis H1 y H4 (Tabla 10) | `validacion/checkpoints_evaluacion/*/*.csv` |
| 2. Generación, modelos | Sonnet 5 vs. Haiku 4.5 (OpenAI denso, prompt estructurado), n = 196 | Tabla 15 | `validacion/evaluacion_generacion_detalle.csv` |
| 3. Generación, pipelines | OpenAI denso vs. BGE-m3 híbrido por pareja de modelo y prompt, n = 196 | Sección 9.5.4 | `validacion/evaluacion_generacion_detalle.csv` |

Las pruebas de **latencia** no están aquí: ya están en `analisis_latencias.ipynb` (reporte en `validacion/reporte_latencias/`).

**Cómo leer los resultados.** Wilcoxon de rangos con signo compara, pregunta por pregunta, los puntajes de dos configuraciones sobre las mismas preguntas. El p-valor dice si la diferencia de medias es distinguible del azar; la diferencia de medias dice qué tan grande es. Las parejas con diferencia exactamente cero se descartan, como hace `scipy` por defecto.

**Comparaciones múltiples.** Cada bloque incluye además el p-valor corregido con el método de Holm dentro de su familia de pruebas. El documento reporta los p-valores sin corregir; la columna corregida sirve para saber qué conclusiones sobreviven si el jurado lo pregunta.

Ejecutar desde la carpeta del proyecto (la que contiene `validacion/`).

In [ ]:
%pip install -q pandas numpy scipy openpyxl

In [27]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

DIR_VALIDACION = Path("./validacion")
DIR_CHECKPOINTS = DIR_VALIDACION / "checkpoints_evaluacion"
RUTA_GENERACION = DIR_VALIDACION / "evaluacion_generacion_detalle.csv"
DIR_SALIDA = DIR_VALIDACION / "reporte_estadistico"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

# Archivos de evaluación RAGAS de cada configuración de retrieve usada en las comparaciones.
# Ojo con las etiquetas heredadas: en los nombres de archivo, "..._hibrido" de OpenAI y
# "bge-m3_con_reranking_cs800_hibrido" son los pipelines MEJORADOS (híbrido + rerank), y
# "bge-m3_cs800_hibrido" es el híbrido simple sin extracción mejorada ni rerank.
RETRIEVAL = {
    "OpenAI denso (cs800)":                "openai/text-embedding-3-large_cs800.csv",
    "OpenAI mejorado (BM25 + rerank, cs800)": "openai/text-embedding-3-large_cs800_hibrido.csv",
    "Gemma denso (cs500)":                 "google/embeddinggemma-300m_cs500.csv",
    "BGE-m3 denso (cs800)":                "BAAI/bge-m3_cs800.csv",
    "BGE-m3 híbrido (cs800)":              "BAAI/bge-m3_cs800_hibrido.csv",
    "BGE-m3 mejorado (híbrido + rerank, cs800)": "BAAI/bge-m3_con_reranking_cs800_hibrido.csv",
}
METRICAS_GEN = ["faithfulness", "answer_relevancy", "answer_correctness", "semantic_similarity"]

In [28]:
def holm(pvalores):
    """Corrección de Holm-Bonferroni; devuelve p ajustados en el mismo orden."""
    p = np.asarray(pvalores, dtype=float)
    orden = np.argsort(p)
    m = len(p)
    ajustado = np.empty(m)
    maximo = 0.0
    for rango, i in enumerate(orden):
        maximo = max(maximo, (m - rango) * p[i])
        ajustado[i] = min(1.0, maximo)
    return ajustado


def wilcoxon_pareado(a, b):
    """a y b: Series alineadas por pregunta. Devuelve diferencia de medias (a - b), p y conteos."""
    ab = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    dif = ab["a"] - ab["b"]
    p = wilcoxon(ab["a"], ab["b"]).pvalue if (dif != 0).any() else 1.0
    return {
        "n": len(ab),
        "media A": ab["a"].mean(),
        "media B": ab["b"].mean(),
        "Diferencia (A − B)": dif.mean(),
        "p": p,
        "Gana A": int((dif > 0).sum()),
        "Gana B": int((dif < 0).sum()),
        "Empates": int((dif == 0).sum()),
    }

## 1. Recuperación: puntaje por pregunta

El puntaje de decisión es el promedio de `llm_context_precision_with_reference` y `context_recall` de cada pregunta, con k = 5 en todas las configuraciones comparadas. Se alinean las configuraciones por el texto de la pregunta (`user_input`).

In [29]:
puntajes = {}
for etiqueta, archivo in RETRIEVAL.items():
    d = pd.read_csv(DIR_CHECKPOINTS / archivo,
                    usecols=["user_input", "llm_context_precision_with_reference", "context_recall"])
    assert d["user_input"].is_unique, f"preguntas repetidas en {archivo}"
    d = d.set_index("user_input")
    puntajes[etiqueta] = (d["llm_context_precision_with_reference"] + d["context_recall"]) / 2

df_ret = pd.DataFrame(puntajes)
print("preguntas:", len(df_ret), "| con algún valor faltante:", int(df_ret.isna().any(axis=1).sum()))
print("promedio (precisión + recall) / 2 por configuración:")
display(df_ret.mean().round(4).to_frame("promedio"))

preguntas: 296 | con algún valor faltante: 0
promedio (precisión + recall) / 2 por configuración:


,promedio
OpenAI denso (cs800),0.7862
"OpenAI mejorado (BM25 + rerank, cs800)",0.7837
Gemma denso (cs500),0.7219
BGE-m3 denso (cs800),0.6667
BGE-m3 híbrido (cs800),0.7186
"BGE-m3 mejorado (híbrido + rerank, cs800)",0.7122


## 2. Recuperación: pruebas pareadas

Hay dos familias. La **decisión de recuperación** reúne las seis comparaciones posibles (todas contra todas) entre las cuatro configuraciones candidatas: la mejor de cada enfoque, es decir OpenAI denso, OpenAI mejorado, Gemma denso (500 tokens) y BGE-m3 híbrido. Las **hipótesis H1 y H4 sobre BGE-m3** reúnen dos comparaciones que contestan si la búsqueda híbrida mejoró al modelo denso (H1) y si el pipeline mejorado aportó algo sobre el híbrido simple (H4). La comparación de H4 para OpenAI (denso contra mejorado) está en la primera familia. Las hipótesis H2 y H3 (cambio de k) no se prueban aquí porque la precisión no es comparable entre distintos k.

In [30]:
# Cada comparación pertenece a una familia: el grupo de pruebas que responde a una misma pregunta.
# Holm se calcula dentro de cada familia.
F_DECISION = "Decisión de recuperación"
F_HIPOTESIS = "Hipótesis H1 y H4 sobre BGE-m3"
PARES_RET = [
    (F_DECISION,  "OpenAI denso (cs800)", "OpenAI mejorado (BM25 + rerank, cs800)"),
    (F_DECISION,  "OpenAI denso (cs800)", "Gemma denso (cs500)"),
    (F_DECISION,  "OpenAI denso (cs800)", "BGE-m3 híbrido (cs800)"),
    (F_DECISION,  "OpenAI mejorado (BM25 + rerank, cs800)", "Gemma denso (cs500)"),
    (F_DECISION,  "OpenAI mejorado (BM25 + rerank, cs800)", "BGE-m3 híbrido (cs800)"),
    (F_DECISION,  "Gemma denso (cs500)",  "BGE-m3 híbrido (cs800)"),
    (F_HIPOTESIS, "BGE-m3 híbrido (cs800)", "BGE-m3 denso (cs800)"),
    (F_HIPOTESIS, "BGE-m3 híbrido (cs800)", "BGE-m3 mejorado (híbrido + rerank, cs800)"),
]
filas = [{"Familia": fam, "A": a, "B": b, **wilcoxon_pareado(df_ret[a], df_ret[b])} for fam, a, b in PARES_RET]
tabla_ret = pd.DataFrame(filas)
tabla_ret["p Holm (dentro de la familia)"] = np.nan
for fam in tabla_ret["Familia"].unique():
    m = tabla_ret["Familia"] == fam
    tabla_ret.loc[m, "p Holm (dentro de la familia)"] = holm(tabla_ret.loc[m, "p"])
display(tabla_ret.round(4))

,Familia,A,B,n,media A,media B,Diferencia (A − B),p,Gana A,Gana B,Empates,p Holm (dentro de la familia)
0,Decisión de recuperación,OpenAI denso (cs800),"OpenAI mejorado (BM25 + rerank, cs800)",296,0.7862,0.7837,0.0026,0.8101,105,112,79,0.9120
1,Decisión de recuperación,OpenAI denso (cs800),Gemma denso (cs500),296,0.7862,0.7219,0.0644,0.0057,124,100,72,0.0172
2,Decisión de recuperación,OpenAI denso (cs800),BGE-m3 híbrido (cs800),296,0.7862,0.7186,0.0676,0.0018,146,107,43,0.0073
3,Decisión de recuperación,"OpenAI mejorado (BM25 + rerank, cs800)",Gemma denso (cs500),296,0.7837,0.7219,0.0618,0.0004,133,97,66,0.0021
4,Decisión de recuperación,"OpenAI mejorado (BM25 + rerank, cs800)",BGE-m3 híbrido (cs800),296,0.7837,0.7186,0.0650,0.0001,144,90,62,0.0008
5,Decisión de recuperación,Gemma denso (cs500),BGE-m3 híbrido (cs800),296,0.7219,0.7186,0.0032,0.4560,127,110,59,0.9120
6,Hipótesis H1 y H4 sobre BGE-m3,BGE-m3 híbrido (cs800),BGE-m3 denso (cs800),296,0.7186,0.6667,0.0519,0.0012,116,84,96,0.0024
7,Hipótesis H1 y H4 sobre BGE-m3,BGE-m3 híbrido (cs800),"BGE-m3 mejorado (híbrido + rerank, cs800)",296,0.7186,0.7122,0.0065,0.3486,26,19,251,0.3486


## 3. Generación: carga y promedio compuesto

El **promedio compuesto** es la media de las cuatro métricas (`faithfulness`, `answer_relevancy`, `answer_correctness`, `semantic_similarity`) de cada respuesta; si una métrica quedó vacía para una respuesta, se promedian las demás.

In [31]:
gen = pd.read_csv(RUTA_GENERACION, usecols=["user_input", "pipeline_retrieve", "modelo_llm", "estilo_prompt"] + METRICAS_GEN)
gen["promedio"] = gen[METRICAS_GEN].mean(axis=1)
print("respuestas:", len(gen), "| preguntas distintas:", gen["user_input"].nunique())
print("valores faltantes por métrica:", gen[METRICAS_GEN].isna().sum().to_dict())
display(gen.groupby(["pipeline_retrieve", "modelo_llm", "estilo_prompt"])[METRICAS_GEN + ["promedio"]].mean().round(4))

def serie(pipeline, modelo, prompt, columna):
    d = gen[(gen["pipeline_retrieve"] == pipeline) & (gen["modelo_llm"] == modelo) & (gen["estilo_prompt"] == prompt)]
    return d.set_index("user_input")[columna]

respuestas: 1568 | preguntas distintas: 196
valores faltantes por métrica: {'faithfulness': 1, 'answer_relevancy': 0, 'answer_correctness': 1, 'semantic_similarity': 0}


faithfulness  \
pipeline_retrieve    modelo_llm       estilo_prompt                 
bge_m3_hibrido_cs800 claude-haiku-4.5 directo              0.9557   
                                      estructurado         0.9643   
                     claude-sonnet-5  directo              0.9679   
                                      estructurado         0.9415   
openai_cs800         claude-haiku-4.5 directo              0.9478   
                                      estructurado         0.9593   
                     claude-sonnet-5  directo              0.9547   
                                      estructurado         0.9408   

                                                     answer_relevancy  \
pipeline_retrieve    modelo_llm       estilo_prompt                     
bge_m3_hibrido_cs800 claude-haiku-4.5 directo                  0.7556   
                                      estructurado             0.7598   
                     claude-sonnet-5  directo                  0.7233   
                                      estructurado             0.7221   
openai_cs800         claude-haiku-4.5 directo                  0.7646   
                                      estructurado             0.7732   
                     claude-sonnet-5  directo                  0.7209   
                                      estructurado             0.7007   

                                                     answer_correctness  \
pipeline_retrieve    modelo_llm       estilo_prompt                       
bge_m3_hibrido_cs800 claude-haiku-4.5 directo                    0.6293   
                                      estructurado               0.6440   
                     claude-sonnet-5  directo                    0.6476   
                                      estructurado               0.6881   
openai_cs800         claude-haiku-4.5 directo                    0.6466   
                                      estructurado               0.6691   
                     claude-sonnet-5  directo                    0.6805   
                                      estructurado               0.7030   

                                                     semantic_similarity  \
pipeline_retrieve    modelo_llm       estilo_prompt                        
bge_m3_hibrido_cs800 claude-haiku-4.5 directo                     0.7747   
                                      estructurado                0.7909   
                     claude-sonnet-5  directo                     0.7985   
                                      estructurado                0.8016   
openai_cs800         claude-haiku-4.5 directo                     0.7812   
                                      estructurado                0.7990   
                     claude-sonnet-5  directo                     0.8005   
                                      estructurado                0.8028   

                                                     promedio  
pipeline_retrieve    modelo_llm       estilo_prompt            
bge_m3_hibrido_cs800 claude-haiku-4.5 directo          0.7788  
                                      estructurado     0.7897  
                     claude-sonnet-5  directo          0.7846  
                                      estructurado     0.7883  
openai_cs800         claude-haiku-4.5 directo          0.7846  
                                      estructurado     0.8002  
                     claude-sonnet-5  directo          0.7891  
                                      estructurado     0.7868

## 4. Generación: Sonnet 5 vs. Haiku 4.5 (Tabla 15)

Misma recuperación (OpenAI denso) y mismo prompt (estructurado); solo cambia el modelo. A = Sonnet, B = Haiku.

In [32]:
filas = []
for m in ["promedio"] + METRICAS_GEN:
    r = wilcoxon_pareado(serie("openai_cs800", "claude-sonnet-5", "estructurado", m),
                         serie("openai_cs800", "claude-haiku-4.5", "estructurado", m))
    filas.append({"Métrica": m, **r})
tabla_llm = pd.DataFrame(filas).rename(columns={"media A": "Sonnet", "media B": "Haiku", "Gana A": "Gana Sonnet", "Gana B": "Gana Haiku"})
tabla_llm["p Holm (5 métricas)"] = holm(tabla_llm["p"])
display(tabla_llm.round(4))

,Métrica,n,Sonnet,Haiku,Diferencia (A − B),p,Gana Sonnet,Gana Haiku,Empates,p Holm (5 métricas)
0,promedio,196,0.7868,0.8002,-0.0133,0.4164,92,101,3,0.8328
1,faithfulness,196,0.9408,0.9593,-0.0185,0.4365,22,25,149,0.8328
2,answer_relevancy,196,0.7007,0.7732,-0.0725,0.0000,59,111,26,0.0000
3,answer_correctness,196,0.7030,0.6691,0.0340,0.0548,105,86,5,0.1645
4,semantic_similarity,196,0.8028,0.7990,0.0038,0.0022,113,77,6,0.0088


## 5. Generación: OpenAI denso vs. BGE-m3 híbrido (sección 9.5.4)

Para cada pareja de modelo y prompt se compara el pipeline de recuperación (A = OpenAI, B = BGE-m3 híbrido) sobre el promedio compuesto. Después se repite sobre el promedio de las cuatro combinaciones por pregunta. Ese último análisis es **exploratorio**: las cuatro combinaciones comparten las mismas preguntas y no son independientes entre sí.

In [33]:
PAREJAS = [("claude-sonnet-5", "estructurado"), ("claude-sonnet-5", "directo"),
           ("claude-haiku-4.5", "estructurado"), ("claude-haiku-4.5", "directo")]
filas = []
for modelo, prompt in PAREJAS:
    r = wilcoxon_pareado(serie("openai_cs800", modelo, prompt, "promedio"),
                         serie("bge_m3_hibrido_cs800", modelo, prompt, "promedio"))
    filas.append({"Modelo y prompt": f"{modelo}, {prompt}", **r})

# Promedio de las cuatro combinaciones por pregunta (exploratorio)
por_pregunta = gen.groupby(["user_input", "pipeline_retrieve"])["promedio"].mean().unstack()
r = wilcoxon_pareado(por_pregunta["openai_cs800"], por_pregunta["bge_m3_hibrido_cs800"])
filas.append({"Modelo y prompt": "Promedio de las 4 combinaciones por pregunta (exploratorio)", **r})
tabla_pipe = pd.DataFrame(filas).rename(columns={"media A": "OpenAI", "media B": "BGE-m3 híbrido", "Gana A": "Gana OpenAI", "Gana B": "Gana BGE-m3"})
tabla_pipe["p Holm (4 parejas)"] = np.nan
tabla_pipe.loc[:3, "p Holm (4 parejas)"] = holm(tabla_pipe.loc[:3, "p"])
display(tabla_pipe.round(4))

,Modelo y prompt,n,OpenAI,BGE-m3 híbrido,Diferencia (A − B),p,Gana OpenAI,Gana BGE-m3,Empates,p Holm (4 parejas)
0,"claude-sonnet-5, estructurado",196,0.7868,0.7883,-0.0015,0.8754,95,96,5,0.8944
1,"claude-sonnet-5, directo",196,0.7891,0.7846,0.0045,0.3437,103,91,2,0.8944
2,"claude-haiku-4.5, estructurado",196,0.8002,0.7897,0.0104,0.1330,103,91,2,0.5319
3,"claude-haiku-4.5, directo",196,0.7846,0.7788,0.0058,0.2981,99,96,1,0.8944
4,Promedio de las 4 combinaciones por pregunta (...,196,0.7902,0.7854,0.0048,0.0256,113,83,0,NaN


### 5.1 Métricas individuales por pareja (apoyo de la sección 9.5.4)

El texto del documento afirma que, entre las métricas individuales, la única diferencia significativa por pareja fue `answer_correctness` con Sonnet y prompt directo (+0.0331, p = 0.013). Esta tabla permite verificarlo: son 16 pruebas (4 parejas × 4 métricas).

In [34]:
filas = []
for (modelo, prompt) in PAREJAS:
    for m in METRICAS_GEN:
        r = wilcoxon_pareado(serie("openai_cs800", modelo, prompt, m), serie("bge_m3_hibrido_cs800", modelo, prompt, m))
        filas.append({"Modelo y prompt": f"{modelo}, {prompt}", "Métrica": m, **r})
tabla_pipe_met = pd.DataFrame(filas).rename(columns={"media A": "OpenAI", "media B": "BGE-m3 híbrido"})
tabla_pipe_met["p Holm (16 pruebas)"] = holm(tabla_pipe_met["p"])
display(tabla_pipe_met[["Modelo y prompt", "Métrica", "OpenAI", "BGE-m3 híbrido", "Diferencia (A − B)", "p", "p Holm (16 pruebas)"]].round(4))
print("Pruebas con p < 0.05 sin corregir:", int((tabla_pipe_met["p"] < 0.05).sum()),
      "| con p Holm < 0.05:", int((tabla_pipe_met["p Holm (16 pruebas)"] < 0.05).sum()))

,Modelo y prompt,Métrica,OpenAI,BGE-m3 híbrido,Diferencia (A − B),p,p Holm (16 pruebas)
0,"claude-sonnet-5, estructurado",faithfulness,0.9408,0.9415,-0.0008,0.7690,1.0000
1,"claude-sonnet-5, estructurado",answer_relevancy,0.7007,0.7221,-0.0214,0.2815,1.0000
2,"claude-sonnet-5, estructurado",answer_correctness,0.7030,0.6881,0.0150,0.5792,1.0000
3,"claude-sonnet-5, estructurado",semantic_similarity,0.8028,0.8016,0.0012,0.3001,1.0000
4,"claude-sonnet-5, directo",faithfulness,0.9547,0.9679,-0.0133,0.1595,1.0000
5,"claude-sonnet-5, directo",answer_relevancy,0.7209,0.7233,-0.0024,0.3358,1.0000
6,"claude-sonnet-5, directo",answer_correctness,0.6807,0.6476,0.0331,0.0133,0.2124
7,"claude-sonnet-5, directo",semantic_similarity,0.8005,0.7985,0.0021,0.2003,1.0000
8,"claude-haiku-4.5, estructurado",faithfulness,0.9593,0.9643,-0.0050,1.0000,1.0000
9,"claude-haiku-4.5, estructurado",answer_relevancy,0.7732,0.7598,0.0134,0.2258,1.0000


Pruebas con p < 0.05 sin corregir: 1 | con p Holm < 0.05: 0


## 6. Exportación

In [35]:
with pd.ExcelWriter(DIR_SALIDA / "pruebas_estadisticas.xlsx") as xw:
    tabla_ret.to_excel(xw, sheet_name="Recuperacion_T12", index=False)
    tabla_llm.to_excel(xw, sheet_name="Sonnet_vs_Haiku_T15", index=False)
    tabla_pipe.to_excel(xw, sheet_name="Pipelines_9_5_4", index=False)
    tabla_pipe_met.to_excel(xw, sheet_name="Pipelines_por_metrica", index=False)
    df_ret.to_excel(xw, sheet_name="Puntajes_retrieval_por_pregunta")
for nombre, t in [("recuperacion_tabla12", tabla_ret), ("sonnet_vs_haiku_tabla15", tabla_llm),
                  ("pipelines_seccion_9_5_4", tabla_pipe), ("pipelines_por_metrica", tabla_pipe_met)]:
    t.to_csv(DIR_SALIDA / f"{nombre}.csv", index=False, encoding="utf-8-sig")
print("Guardado en", DIR_SALIDA.resolve())

Guardado en C:\Users\gomex\OneDrive\Escritorio\TG - Pruebas 2\validacion\reporte_estadistico
